In [ ]:
## This notebook makes predictions over test images using trained predictor

In [ ]:
# import packages
import os, cv2
import numpy as np
import pandas as pd
from datetime import datetime
## packages for deep learning
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image

## check GPU
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Current GPU: {torch.cuda.get_device_name(0)}")
    device = torch.device("cuda:0")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

In [ ]:
# load the pretrained model
THRESH = 20
n_c = 25
path_wd = '../' ## set working directory
if not os.path.isdir(path_wd + 'output/npy/npy_' + str(THRESH) + '_' + str(n_c)):
    os.mkdir(path_wd + 'output/npy/npy_' + str(THRESH) + '_' + str(n_c))

## Define model architecture (must match training)
class APLModel(nn.Module):
    def __init__(self, num_classes=3):
        super(APLModel, self).__init__()
        # Load pretrained ResNet-50
        resnet = models.resnet50(pretrained=False)  # Don't load pretrained weights
        # Remove final FC layer (keep until avgpool)
        self.features = nn.Sequential(*list(resnet.children())[:-1])
        # GlobalAveragePooling is already included in ResNet's avgpool
        self.fc = nn.Linear(2048, num_classes)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        x = self.features(x)        # [batch, 2048, 1, 1]
        x = torch.flatten(x, 1)     # [batch, 2048]
        x = self.fc(x)              # [batch, 3]
        x = self.sigmoid(x)         # [batch, 3]
        return x

time_start = datetime.now()
print('Start:', time_start)

## Load model
model_resnet = APLModel(num_classes=3).to(device)
checkpoint = torch.load(path_wd + 'output/models/APL_' + str(THRESH) + '_' + str(n_c) + '.pth', 
                       map_location=device)
model_resnet.load_state_dict(checkpoint['model_state_dict'])
model_resnet.eval()  # Set to evaluation mode

print('Finished:', datetime.now() - time_start)

In [ ]:
# loop over the shadow_free folder
folder_name = path_wd + 'data/test/'
file_list = os.listdir(folder_name)
rs = 10
size_wind = 100
num_col = int(10000/rs)

## Define preprocessing (equivalent to Keras preprocess_input)
preprocess = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize(100),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                       std=[0.229, 0.224, 0.225])
])

for img_name in file_list:
    print(img_name)
    probs_mat = np.zeros((num_col, num_col, 3))
    img_path = folder_name + img_name
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) ## transform color channel
    
    ## start prediction
    time_start = datetime.now()
    ## use the pretrained model as convolution kernel
    with torch.no_grad():  # Disable gradient computation
        for i in range(num_col):
            r_1 = int(np.max([0, i*rs + rs/2 - size_wind/2]))
            r_2 = int(np.min([10000, i*rs + rs/2 +  size_wind/2]))
            
            # Prepare batch for this row
            batch_tensors = []
            for j in range(num_col):
                c_1 = int(np.max([0, j*rs + rs/2 - size_wind/2]))
                c_2 = int(np.min([10000, j*rs + rs/2 + size_wind/2]))
                img_patch = img[r_1:r_2, c_1:c_2]
                
                if img_patch.shape != (100, 100, 3):
                    img_patch = cv2.resize(img_patch, (100, 100)) ## boundary and corner patches
                
                # Preprocess and convert to tensor
                img_tensor = preprocess(img_patch)
                batch_tensors.append(img_tensor)
            
            # Stack into batch
            x_img = torch.stack(batch_tensors).to(device)  # [num_col, 3, 100, 100]
            
            # Forward pass
            outputs = model_resnet(x_img)  # [num_col, 3]
            probs_mat[i] = outputs.cpu().numpy()  # Convert back to numpy
    
    npy_path = path_wd + 'output/npy/npy_' + str(THRESH) + '_' + str(n_c) + '/' + img_name[:(-4)] + '.npy'
    np.save(npy_path, probs_mat)
    print('Time:', datetime.now() - time_start)

print('All finished!')